# 01 — Signed Envelopes and the Secure Bus

Every message in specter-1 is an ECDSA-signed envelope. This notebook walks through:

1. Generating a keypair, sealing an envelope, sending it over an `InProcessBus`.
2. The five rejection categories (`bad_signature`, `replay`, `unknown_sender`, `version_mismatch`, plus identity-layer rejections from notebook 02).
3. Why canonical-JSON serialization matters for signature verification.

**Cell types:** *intuition* (build mental model), *claim* (assert measured behavior), *limit* (residual gap).

In [1]:
from specter.crypto import Keypair
from specter.messages import KIND_POSE, PoseReport, decode, encode
from specter.secure_bus import (
    Envelope, Identity, InProcessBus, ReplayWindow, Roster,
    VerificationError, envelope_from_wire, envelope_to_wire, open_envelope,
)

## Intuition: seal one envelope and verify it

An `Identity` is `(agent_id, Keypair)`. `Identity.seal(kind, payload)` produces an `Envelope` with a signature, nonce, and timestamp. The receiver looks up the sender's pubkey in the `Roster` and verifies.

In [2]:
alpha = Identity('alpha', Keypair.generate())
bravo = Identity('bravo', Keypair.generate())
roster = Roster()
roster.add(alpha.agent_id, alpha.keypair.public_bytes)
roster.add(bravo.agent_id, bravo.keypair.public_bytes)

pose = PoseReport(agent_id='alpha', x=1.0, y=2.0, theta=0.0, timestamp_ns=1_000_000_000)
env = alpha.seal(KIND_POSE, encode(pose), timestamp_ns=1_000_000_000)
print(f'kind={env.kind}  sender={env.sender_id}  nonce={env.nonce}  bytes={len(env.signature)}b sig')

kind=pose_report  sender=alpha  nonce=1  bytes=72b sig


In [3]:
# Receive: open_envelope verifies the signature against the roster + a per-sender ReplayWindow.
replay = ReplayWindow()
payload = open_envelope(env, roster, replay)
decoded = decode(env.kind, payload)
print(f'verified pose: x={decoded.x} y={decoded.y} t_ns={decoded.timestamp_ns}')

verified pose: x=1.0 y=2.0 t_ns=1000000000


## Claim: the receive pipeline rejects 4 distinct attack categories

Each rejection category has its own `VerificationError` substring; the trust engine routes them to typed `AnomalyEvent`s. The evidence behind this notebook lives in `tests/test_secure_bus.py` — it exhaustively exercises every category.

In [4]:
categories_seen = []

# 1) bad_signature — sign with a key the roster doesn't know
stranger = Identity('alpha', Keypair.generate())  # same id, different key
evil_env = stranger.seal(KIND_POSE, encode(pose), timestamp_ns=2_000_000_000)
try:
    open_envelope(evil_env, roster, ReplayWindow())
except VerificationError as e:
    categories_seen.append(('bad_signature', str(e)))

# 2) replay — re-submit the same nonce
fresh_replay = ReplayWindow()
open_envelope(env, roster, fresh_replay)  # first ok
try:
    open_envelope(env, roster, fresh_replay)  # second triggers replay
except VerificationError as e:
    categories_seen.append(('replay', str(e)))

# 3) unknown_sender — sender not in roster
ghost = Identity('ghost', Keypair.generate())
ghost_env = ghost.seal(KIND_POSE, encode(pose), timestamp_ns=3_000_000_000)
try:
    open_envelope(ghost_env, roster, ReplayWindow())
except VerificationError as e:
    categories_seen.append(('unknown_sender', str(e)))

# 4) version_mismatch — handcraft an envelope with a future version byte
import dataclasses
future_env = dataclasses.replace(env, version=99, nonce=env.nonce + 1)
try:
    open_envelope(future_env, roster, ReplayWindow())
except VerificationError as e:
    categories_seen.append(('version_mismatch', str(e)))

for cat, msg in categories_seen:
    print(f'{cat:18s}  {msg}')
assert len(categories_seen) == 4, 'expected 4 distinct rejection categories'

bad_signature       bad signature from alpha
replay              replay from alpha nonce=1
unknown_sender      unknown sender ghost
version_mismatch    unsupported version 99


## Intuition: one bus, multiple subscribers

An `InProcessBus` lets multiple agents subscribe to the same topic. Each subscriber decides independently whether to accept or reject an envelope — a structural property that survives any bus swap (LossyBus, Sros2Bus).

In [5]:
bus = InProcessBus()
received = []

def make_listener(name):
    replay = ReplayWindow()
    def on_wire(wire):
        e = envelope_from_wire(wire)
        try:
            payload = open_envelope(e, roster, replay)
            received.append((name, 'ok', e.sender_id))
        except VerificationError as err:
            received.append((name, 'reject', str(err)[:30]))
    return on_wire

bus.subscribe('pose', make_listener('bravo'))
bus.subscribe('pose', make_listener('charlie'))

# Alpha publishes one envelope; both subscribers should receive it
fresh_env = alpha.seal(KIND_POSE, encode(pose), timestamp_ns=4_000_000_000)
bus.publish('pose', envelope_to_wire(fresh_env))
for entry in received:
    print(entry)

('bravo', 'ok', 'alpha')
('charlie', 'ok', 'alpha')


## Limit

Envelope-level signing protects bus contents but does **not** protect against a compromised hardware key. If a robot's signing key is stolen off the device, the attacker can sign valid-looking envelopes with that identity. The defense for that case is the **hardware attestation** layer (notebook 02, ADR 0009) — and is currently a mock implementation. The real TPM/Secure-Enclave integration is Phase 4 per `docs/HARDWARE_READINESS.md`.

See `docs/THREAT_MODEL.md` § *hardware-key-compromise* for the residual-risk framing.